# Move Indexing And Legal Mask Smoke Test

This notebook inspects the fixed 4672-action policy space and legal move mask. It is for quick local sanity checks, not reportable experiments.

In [1]:
from pathlib import Path
import sys

project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent

src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

In [2]:
import chess

from mcchess.board import (
    MOVE_PLANES,
    POLICY_SIZE,
    index_to_move,
    legal_policy_mask,
    move_to_index,
)

## Policy Space

In [3]:
{
    "move_planes_per_square": MOVE_PLANES,
    "policy_size": POLICY_SIZE,
    "formula": "index = from_square * 73 + move_plane",
}

{'move_planes_per_square': 73,
 'policy_size': 4672,
 'formula': 'index = from_square * 73 + move_plane'}

## Initial Position

In [4]:
board = chess.Board()
mask = legal_policy_mask(board)

{
    "fen": board.fen(),
    "legal_moves": board.legal_moves.count(),
    "mask_shape": mask.shape,
    "mask_sum": int(mask.sum()),
    "mask_dtype": str(mask.dtype),
}

{'fen': 'rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1',
 'legal_moves': 20,
 'mask_shape': (4672,),
 'mask_sum': 20,
 'mask_dtype': 'float32'}

In [5]:
initial_examples = []
for move in list(board.legal_moves)[:10]:
    index = move_to_index(board, move)
    initial_examples.append(
        {
            "uci": move.uci(),
            "index": index,
            "decoded": index_to_move(board, index).uci(),
        }
    )

initial_examples

[{'uci': 'g1h3', 'index': 494, 'decoded': 'g1h3'},
 {'uci': 'g1f3', 'index': 501, 'decoded': 'g1f3'},
 {'uci': 'b1c3', 'index': 129, 'decoded': 'b1c3'},
 {'uci': 'b1a3', 'index': 136, 'decoded': 'b1a3'},
 {'uci': 'h2h3', 'index': 1095, 'decoded': 'h2h3'},
 {'uci': 'g2g3', 'index': 1022, 'decoded': 'g2g3'},
 {'uci': 'f2f3', 'index': 949, 'decoded': 'f2f3'},
 {'uci': 'e2e3', 'index': 876, 'decoded': 'e2e3'},
 {'uci': 'd2d3', 'index': 803, 'decoded': 'd2d3'},
 {'uci': 'c2c3', 'index': 730, 'decoded': 'c2c3'}]

## Castling

In [6]:
castle_board = chess.Board("r3k2r/8/8/8/8/8/8/R3K2R w KQkq - 0 1")

[
    {
        "uci": uci,
        "index": move_to_index(castle_board, chess.Move.from_uci(uci)),
        "decoded": index_to_move(
            castle_board,
            move_to_index(castle_board, chess.Move.from_uci(uci)),
        ).uci(),
    }
    for uci in ("e1g1", "e1c1")
]

[{'uci': 'e1g1', 'index': 307, 'decoded': 'e1g1'},
 {'uci': 'e1c1', 'index': 335, 'decoded': 'e1c1'}]

## Promotions

In [7]:
promotion_board = chess.Board("8/P7/8/8/8/8/8/k6K w - - 0 1")

[
    {
        "uci": uci,
        "index": move_to_index(promotion_board, chess.Move.from_uci(uci)),
        "decoded": index_to_move(
            promotion_board,
            move_to_index(promotion_board, chess.Move.from_uci(uci)),
        ).uci(),
    }
    for uci in ("a7a8q", "a7a8n", "a7a8b", "a7a8r")
]

[{'uci': 'a7a8q', 'index': 3504, 'decoded': 'a7a8q'},
 {'uci': 'a7a8n', 'index': 3568, 'decoded': 'a7a8n'},
 {'uci': 'a7a8b', 'index': 3571, 'decoded': 'a7a8b'},
 {'uci': 'a7a8r', 'index': 3574, 'decoded': 'a7a8r'}]

## En Passant

In [8]:
ep_board = chess.Board()
for uci in ("e2e4", "a7a6", "e4e5", "d7d5"):
    ep_board.push(chess.Move.from_uci(uci))

ep_move = chess.Move.from_uci("e5d6")
ep_index = move_to_index(ep_board, ep_move)

{
    "fen": ep_board.fen(),
    "move": ep_move.uci(),
    "is_en_passant": ep_board.is_en_passant(ep_move),
    "index": ep_index,
    "decoded": index_to_move(ep_board, ep_index).uci(),
    "mask_value": float(legal_policy_mask(ep_board)[ep_index]),
}

{'fen': 'rnbqkbnr/1pp1pppp/p7/3pP3/8/8/PPPP1PPP/RNBQKBNR w KQkq d6 0 3',
 'move': 'e5d6',
 'is_en_passant': True,
 'index': 2677,
 'decoded': 'e5d6',
 'mask_value': 1.0}